# Darcy equation: exercise 4

Let $\Omega=(0,1)^2$ with boundary $\partial \Omega$ and outward unit normal ${\nu}$. Given 
$k=I$ the matrix permeability and $f$ a scalar source term, we want to solve the following problem: find $({q}, p)$ such that
$$
\left\{
\begin{array}{ll}
\begin{array}{l} 
k^{-1} {q} + \nabla p = 0\\
\nabla \cdot {q} = f
\end{array}
&\text{in } \Omega
\end{array}
\right.
$$
with boundary conditions:
$$ \nu \cdot q = 0 \text{ on } \partial \Omega$$
Where $f = \pm 1$ represents two wells, one with $+1$ and one with $-1$, set on the two corners of the domain.
The problem clearly does not have a unique solution and we need to design a strategy to make it solvable.

This is the guided ("fill in the code") version of `ex4.ipynb` -- work through the cells in order, completing each `__TODO__`. Compare against `ex4.ipynb` once you're done, or if you get stuck.

First we import some of the standard modules.

In [ ]:
import numpy as np
import scipy.sparse as sps

import porepy as pp
import pygeon as pg

We create now the grid, since we will use a Raviart-Thomas approximation for ${q}$ we are restricted to simplices. In this example we consider a 2-dimensional grid.

In [ ]:
mesh_size = 0.05
# creation of the grid
sd = pg.unit_grid(2, mesh_size, as_mdg=False)
# compute the geometrical properties of the grid
sd.compute_geometry()

Let us declare the finite element spaces that we are going to use

In [ ]:
key = "flow"

# declare the discretization objects, useful to setup the data
rt0 = pg.RT0(key)
p0 = pg.PwConstants(key)

# build the degrees of freedom
dofs = np.array([rt0.ndof(sd), p0.ndof(sd)])

With the following code we set the data, in particular the permeability tensor and the boundary conditions. Since we need to identify each side of $\partial \Omega$ we need few steps.

In [ ]:
# TODO: select the well cells GEOMETRICALLY, not by hardcoded index (that
# breaks the moment the grid/mesh changes). For each of the two target corners,
# (0, 0) and (1, 1), find the cell whose center (sd.cell_centers[:2, :]) is
# closest to it
def closest_cell(sd, point):
    dist = np.linalg.norm(
        sd.cell_centers[:2, :] - np.asarray(point).reshape(-1, 1), axis=0
    )
    return np.argmin(dist)


well_in = __TODO__
well_out = __TODO__

# set the permeability
inv_perm = pp.SecondOrderTensor(np.ones(sd.num_cells))
param = {pg.SECOND_ORDER_TENSOR: inv_perm}
data = pp.initialize_data({}, key, param)

# TODO: build scalar_source (length sd.num_cells): +1 at well_in, -1 at well_out, 0 elsewhere
scalar_source = __TODO__

# the whole boundary is a natural (flux) condition here
ess_q = np.zeros(dofs[0], dtype=bool)
ess_p = np.zeros(dofs[1], dtype=bool)
bc_ess = np.hstack((ess_q, ess_p))

Once the data are assigned to the grid, we construct the matrices. In particular, the linear system associated with the equation is given as
$$
\left(
\begin{array}{cc} 
A & -B^\top\\
B & 0
\end{array}
\right)
\left(
\begin{array}{c} 
q\\ 
p
\end{array}
\right)
=\left(
\begin{array}{c} 
0\\ 
f
\end{array}
\right)
$$<br>
To construct the saddle-point problem, we rely on the `scipy.sparse` function `block_array`. Once the matrix is created, we also construct the right-hand side containing the source term.

In [ ]:
# construct the local matrices
A = rt0.assemble_mass_matrix(sd, data)
mass_p0 = p0.assemble_mass_matrix(sd)
B = mass_p0 @ rt0.assemble_diff_matrix(sd)

# TODO: assemble the saddle point problem with sps.block_array
spp = __TODO__

However, the matrix `spp` is singular but we can impose that the pressure has zero average
$$
    \int_\Omega p = 0 \quad \Rightarrow \quad \sum_i p_i = 0
$$
since our pressure degrees of freedom already includes the measure of the cells. We can use a Lagrange multiplier to impose this constraint, we add a new line to the system and its corresponding (anti)transpose.

In [ ]:
# TODO: build the constraint row cons (shape (1, dofs.sum())): 1 on every
# pressure dof, 0 on every flux dof, so that cons @ [q, p] == sum(p)
cons = __TODO__

# add the constraint
spp = sps.block_array(
    [
        [spp, -cons.T],
        [cons, None],
    ],
    format="csc",
)

# TODO: assemble the right-hand side (scalar_source in the pressure block,
# 0 for the Lagrange multiplier row)
rhs = np.zeros(dofs.sum() + 1)
rhs[dofs[0] : -1] += __TODO__

We need to solve the linear system and extract the two solutions $q$ and $p$, by remembering to discard the last row used for the Lagrange multiplier.

In [ ]:
# fix the bc
bc = np.zeros_like(rhs, dtype=bool)
bc[:-1] = bc_ess

# solve the problem
ls = pg.LinearSystem(spp, rhs)
ls.flag_ess_bc(bc, np.zeros_like(rhs))
x = ls.solve()[:-1]

# TODO: split the solution into the components q and p
idx = np.cumsum(dofs[:-1])
q, p = __TODO__

Since the computed $q$ is one value per facet of the grid, for visualization purposes we project the flux in each cell center as vector. We finally export the solution to be visualized by [ParaView](https://www.paraview.org/).

In [ ]:
# post process variables
proj_q = rt0.eval_at_cell_centers(sd)
# TODO: project q to cell centers and reshape to (pg.AMBIENT_DIM, -1)
cell_q = __TODO__
cell_p = p0.eval_at_cell_centers(sd) @ p

save = pp.Exporter(sd, "sol", folder_name="ex4")
save.write_vtu([("cell_p", cell_p), ("cell_q", cell_q)])

We verify that the computed solution matches the expected reference values.

In [ ]:
# Consistency check -- once your implementation is correct, this should pass
assert np.isclose(np.linalg.norm(cell_p), 0.00015528953981672954)
assert np.isclose(np.linalg.norm(cell_q), 0.009705132398026632)